## 3. Transformation

This phase involves cleaning, standardizing, and applying necessary business logic to the extracted raw data.

**Key transformation tasks**

- **Currency conversion:** Merge latest currency exchange rates to calculate local prices.
- **Delivery metrics:** Add columns for late deliveries and latency days.
- **Locality flag:** Determine if customers are local relative to stores.
- **Lookup tables:** Resolve ambiguous columns (e.g. order statuses).
- **Transformed data staging:** Write outputs to `staging_2`.


In [23]:
import pandas as pd
import numpy as np
from datetime import datetime

In [24]:
customers  = pd.read_csv(r'D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Transformation\staging_1\customers.csv')
exchange    = pd.read_csv(r'D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Transformation\staging_1\exchange_rates.csv')
order_items = pd.read_csv(r'D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Transformation\staging_1\order_items.csv')
orders      = pd.read_csv(r'D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Transformation\staging_1\orders.csv')
products    = pd.read_csv(r'D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Transformation\staging_1\products.csv')
staffs      = pd.read_csv(r'D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Transformation\staging_1\staffs.csv')
stocks      = pd.read_csv(r'D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Transformation\staging_1\stocks.csv')

print("All files loaded successfully")
print("=" * 60)

All files loaded successfully


In [25]:
# ─────────────────────────────────────────
# TASK 1 — LOOKUP TABLE: Order Status
# ───────────────────────────────────────
order_status_lookup = pd.DataFrame({
    'order_status'      : [1, 2, 3, 4],
    'order_status_name' : ['Pending', 'Processing', 'Rejected', 'Completed']
})

# Merge into orders
orders = orders.merge(order_status_lookup, on='order_status', how='left')

print(order_status_lookup.to_string(index=False))
print(f"\norders['order_status_name'] sample:")
print(orders[['order_id', 'order_status', 'order_status_name']].head(3).to_string(index=False))

 order_status order_status_name
            1           Pending
            2        Processing
            3          Rejected
            4         Completed

orders['order_status_name'] sample:
 order_id  order_status order_status_name
        1             4         Completed
        2             4         Completed
        3             4         Completed


In [26]:
print("\n" + "=" * 60)
print("\n🚚 TASK 2 — Delivery Metrics")

# Convert date columns
orders['order_date']    = pd.to_datetime(orders['order_date'])
orders['required_date'] = pd.to_datetime(orders['required_date'])
orders['shipped_date']  = pd.to_datetime(orders['shipped_date'])

# Latency Days = shipped_date - order_date
orders['latency_days'] = (orders['shipped_date'] - orders['order_date']).dt.days

# Late Delivery = shipped_date > required_date
orders['is_late_delivery'] = (orders['shipped_date'] > orders['required_date']).astype(int)
# 1 = Late, 0 = On Time

# Delivery Status label
orders['delivery_status'] = orders['is_late_delivery'].map({
    0 : 'On Time',
    1 : 'Late'
})

print(orders[['order_id', 'order_date', 'required_date',
              'shipped_date', 'latency_days',
              'is_late_delivery', 'delivery_status']].head(5).to_string(index=False))

late_count = orders['is_late_delivery'].sum()
total      = len(orders)
print(f"\nTotal Orders   : {total}")
print(f"Late Deliveries: {late_count} ({late_count/total*100:.1f}%)")
print(f"On Time        : {total - late_count} ({(total-late_count)/total*100:.1f}%)")
print(f"Avg Latency    : {orders['latency_days'].mean():.1f} days")



🚚 TASK 2 — Delivery Metrics
 order_id order_date required_date shipped_date  latency_days  is_late_delivery delivery_status
        1 2016-01-01    2016-01-03   2016-01-03             2                 0         On Time
        2 2016-01-01    2016-01-04   2016-01-03             2                 0         On Time
        3 2016-01-02    2016-01-05   2016-01-03             1                 0         On Time
        4 2016-01-03    2016-01-04   2016-01-05             2                 1            Late
        5 2016-01-03    2016-01-06   2016-01-06             3                 0         On Time

Total Orders   : 1445
Late Deliveries: 458 (31.7%)
On Time        : 987 (68.3%)
Avg Latency    : 2.0 days


In [27]:
print("\n" + "=" * 60)
print("\n📍 TASK 3 — Locality Flag")

# Store locations lookup
store_lookup = pd.DataFrame({
    'store_id'   : [1, 2, 3],
    'store_name' : ['Santa Cruz Bikes', 'Baldwin Bikes', 'Rowlett Bikes'],
    'store_city' : ['Santa Cruz', 'Baldwin', 'Rowlett'],
    'store_state': ['CA', 'NY', 'TX']
})

# Merge store state into orders
orders = orders.merge(store_lookup[['store_id', 'store_state', 'store_name']],
                      on='store_id', how='left')

# Merge customer state into orders
orders = orders.merge(customers[['customer_id', 'state']].rename(
                      columns={'state': 'customer_state'}),
                      on='customer_id', how='left')

# Local = customer state matches store state
orders['is_local_customer'] = (
    orders['customer_state'] == orders['store_state']
).astype(int)
# 1 = Local, 0 = Non-Local

orders['locality'] = orders['is_local_customer'].map({
    1 : 'Local',
    0 : 'Non-Local'
})

print(orders[['order_id', 'store_name', 'store_state',
              'customer_state', 'is_local_customer', 'locality']].head(5).to_string(index=False))

local_count = orders['is_local_customer'].sum()
print(f"\nLocal Customers    : {local_count} ({local_count/total*100:.1f}%)")
print(f"Non-Local Customers: {total-local_count} ({(total-local_count)/total*100:.1f}%)")



📍 TASK 3 — Locality Flag
 order_id       store_name store_state customer_state  is_local_customer locality
        1 Santa Cruz Bikes          CA             CA                  1    Local
        2    Baldwin Bikes          NY             NY                  1    Local
        3    Baldwin Bikes          NY             NY                  1    Local
        4 Santa Cruz Bikes          CA             CA                  1    Local
        5    Baldwin Bikes          NY             NY                  1    Local

Local Customers    : 1445 (100.0%)
Non-Local Customers: 0 (0.0%)


In [28]:
# ─────────────────────────────────────────
# TASK 4 — CURRENCY CONVERSION
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("\n💱 TASK 4 — Currency Conversion")

# Clean exchange rates
exchange = exchange[['Currency', 'Rate']].copy()

# Calculate revenue in order_items
order_items['net_revenue_usd'] = (
    order_items['quantity'] *
    order_items['list_price'] *
    (1 - order_items['discount'])
).round(2)

# Add key currencies as new columns
key_currencies = ['EGP', 'EUR', 'GBP', 'AED', 'SAR']

for currency in key_currencies:
    rate = exchange.loc[exchange['Currency'] == currency, 'Rate'].values
    if len(rate) > 0:
        order_items[f'net_revenue_{currency}'] = (
            order_items['net_revenue_usd'] * rate[0]
        ).round(2)

print(order_items[['order_id', 'product_id', 'quantity',
                    'list_price', 'discount', 'net_revenue_usd',
                    'net_revenue_EGP', 'net_revenue_EUR']].head(5).to_string(index=False))

print(f"\nExchange Rates Used:")
for currency in key_currencies:
    rate = exchange.loc[exchange['Currency'] == currency, 'Rate'].values
    if len(rate) > 0:
        print(f"  1 USD = {rate[0]:.4f} {currency}")



💱 TASK 4 — Currency Conversion
 order_id  product_id  quantity  list_price  discount  net_revenue_usd  net_revenue_EGP  net_revenue_EUR
        1          20         1      599.99      0.20           479.99         25689.19           414.53
        1           8         2     1799.99      0.07          3347.98        179184.77          2891.42
        1          10         2     1549.00      0.05          2943.10        157515.49          2541.76
        1          16         2      599.99      0.05          1139.98         61012.03           984.52
        1           4         1     2899.99      0.20          2319.99        124166.47          2003.62

Exchange Rates Used:
  1 USD = 53.5203 EGP
  1 USD = 0.8636 EUR
  1 USD = 0.7518 GBP
  1 USD = 3.6732 AED
  1 USD = 3.7536 SAR


In [30]:
# ─────────────────────────────────────────
# TASK 5 — SAVE STAGED TRANSFORMED DATA
# ─────────────────────────────────────────
import os
OUTPUT_DIR = r'D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Transformation\staging_2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

staffs.to_csv(os.path.join(OUTPUT_DIR, 'staffs.csv'), index=False)
stocks.to_csv(os.path.join(OUTPUT_DIR, 'stocks.csv'), index=False)
products.to_csv(os.path.join(OUTPUT_DIR, 'products.csv'), index=False)
customers.to_csv(os.path.join(OUTPUT_DIR, 'customers.csv'), index=False)
orders.to_csv(os.path.join(OUTPUT_DIR, 'orders.csv'), index=False)
order_items.to_csv(os.path.join(OUTPUT_DIR, 'order_items.csv'), index=False)
order_status_lookup.to_csv(os.path.join(OUTPUT_DIR, 'order_status_lookup.csv'), index=False)
store_lookup.to_csv(os.path.join(OUTPUT_DIR, 'store_lookup.csv'), index=False)
